In [ ]:
import concurrent.futures
import os
import time

import psycopg2
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

DB_CONFIG = {
    "dbname": "lab2BDsem2",
    "user": "postgres",
    "password": os.getenv("DB_PASSWORD"),
    "host": "localhost",
    "port": "5432",
}

TOTAL_EXPECTED = 100000
THREAD_SCENARIOS = [1, 5, 10, 20]

SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS user_counter (
    USER_ID INT PRIMARY KEY,
    Counter INT,
    Version INT
);
"""

SEED_SQL = """
INSERT INTO user_counter (USER_ID, Counter, Version)
VALUES (1, 0, 0)
ON CONFLICT (USER_ID) DO UPDATE
SET Counter = EXCLUDED.Counter,
    Version = EXCLUDED.Version;
"""

def get_connection():
    return psycopg2.connect(**DB_CONFIG)


def prepare_table():
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(SCHEMA_SQL)
        cursor.execute(SEED_SQL)
        conn.commit()
    finally:
        conn.close()

def reset_counter():
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute("UPDATE user_counter SET Counter = 0, Version = 0 WHERE USER_ID = 1;")
        conn.commit()
    finally:
        conn.close()

def get_final_counter():
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT Counter FROM user_counter WHERE USER_ID = 1;")
        return cursor.fetchone()[0]
    finally:
        conn.close()

def lost_update(iterations):
    conn = get_connection()
    try:
        cursor = conn.cursor()
        for _ in range(iterations):
            cursor.execute("SELECT Counter FROM user_counter WHERE USER_ID = 1;")
            counter = cursor.fetchone()[0]
            counter += 1
            cursor.execute("UPDATE user_counter SET Counter = %s WHERE USER_ID = 1;", (counter,))
            conn.commit()
    finally:
        conn.close()

def worker_in_place_update(iterations):
    conn = get_connection()
    try:
        cursor = conn.cursor()
        for _ in range(iterations):
            cursor.execute("UPDATE user_counter SET Counter = Counter + 1 WHERE USER_ID = 1;")
            conn.commit()
    finally:
        conn.close()

def worker_row_level_locking(iterations):
    conn = get_connection()
    try:
        cursor = conn.cursor()
        for _ in range(iterations):
            cursor.execute("SELECT Counter FROM user_counter WHERE USER_ID = 1 FOR UPDATE;")
            counter = cursor.fetchone()[0]
            counter += 1
            cursor.execute("UPDATE user_counter SET Counter = %s WHERE USER_ID = 1;", (counter,))
            conn.commit()
    finally:
        conn.close()

def worker_optimistic(iterations):
    conn = get_connection()
    try:
        cursor = conn.cursor()
        for _ in range(iterations):
            while True:
                cursor.execute("SELECT Counter, Version FROM user_counter WHERE USER_ID = 1;")
                counter, version = cursor.fetchone()

                counter += 1
                new_version = version + 1

                cursor.execute(
                    """
                    UPDATE user_counter
                    SET Counter = %s, Version = %s
                    WHERE USER_ID = 1 AND Version = %s;
                    """,
                    (counter, new_version, version),
                )

                rowcount = cursor.rowcount
                conn.commit()

                if rowcount > 0:
                    break
    finally:
        conn.close()

def run_test(target_function, test_name, threads, iterations):
    reset_counter()
    start_time = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=threads) as executor:
        futures = [executor.submit(target_function, iterations) for _ in range(threads)]
        concurrent.futures.wait(futures)

    end_time = time.time()
    final_counter = get_final_counter()
    expected_counter = threads * iterations

    print(test_name)
    print(f"Execution time: {end_time - start_time:.3f} seconds")
    print(f"Final counter: {final_counter}")
    print(f"Expected counter: {expected_counter}")
    print(f"Correct: {final_counter == expected_counter}")
    print("-" * 40)

def run_all_load_scenarios():
    prepare_table()

    for threads in THREAD_SCENARIOS:
        if TOTAL_EXPECTED % threads != 0:
            raise ValueError(f"TOTAL_EXPECTED={TOTAL_EXPECTED} is not divisible by threads={threads}")

        iterations = TOTAL_EXPECTED // threads

        print(
            f"\n{threads} threads x {iterations} iterations "
            f"(expected: {TOTAL_EXPECTED}) ==="
        )

        run_test(lost_update, "Lost-update", threads, iterations)
        run_test(worker_in_place_update, "In-place update", threads, iterations)
        run_test(worker_row_level_locking, "Row-level locking", threads, iterations)
        run_test(worker_optimistic, "Optimistic concurrency control", threads, iterations)

run_all_load_scenarios()



1 threads x 100000 iterations (expected: 100000) ===
Lost-update
Execution time: 74.406 seconds
Final counter: 100000
Expected counter: 100000
Correct: True
----------------------------------------
In-place update
Execution time: 47.070 seconds
Final counter: 100000
Expected counter: 100000
Correct: True
----------------------------------------
Row-level locking
Execution time: 61.637 seconds
Final counter: 100000
Expected counter: 100000
Correct: True
----------------------------------------
Optimistic concurrency control
Execution time: 76.020 seconds
Final counter: 100000
Expected counter: 100000
Correct: True
----------------------------------------

5 threads x 20000 iterations (expected: 100000) ===
Lost-update
Execution time: 46.727 seconds
Final counter: 21220
Expected counter: 100000
Correct: False
----------------------------------------
In-place update
Execution time: 37.982 seconds
Final counter: 100000
Expected counter: 100000
Correct: True
-------------------------------